In [0]:
file = "/Volumes/workspace/default/learning/payments.csv"

In [0]:
payment = spark.read.csv(file, header= True, inferSchema=True)
display(payment)

In [0]:
payment.write.mode("overwrite").format("delta").saveAsTable("payment")

In [0]:
display(spark.table("payment"))

In [0]:
%sql
update payment
set status = 'completed'
where transaction_id = 'TXN002';

In [0]:
%sql
select * from payment order by transaction_id;

In [0]:
%sql
insert into payment values
 ('TXN021', 'C104', 320.00, 'completed', '2024-12-01 16:30:00'),
 ('TXN022', 'C100', 55.50, 'pending', '2024-12-01 17:00:00');


In [0]:
%sql
describe history payment;

In [0]:
%sql
select * from payment version as of 0;

RESTORE creates a new version by copying data from the old version. The old versions still exist in history 

This is safer than directly manipulating files.


In [0]:
%sql
restore table payment to version as of 3;
select * from payment

In [0]:
%sql
create or replace temp view incoming_payment as 
select * from values 
('TXN003', 'C100', 245.50, 'refunded', '2024-12-01 12:00:00'),
('TXN023', 'C105', 175.00, 'completed', '2024-12-01 18:00:00')
AS t(transaction_id, customer_id, amount, status, created_at);
merge into payment as target
using incoming_payment as source
on target.transaction_id = source.transaction_id
when matched then
update set target.status = source.status
when not matched then
insert *;